In [1]:
%%writefile tools/cache.py
"""Tiny disk cache for search results (Day 25). Off unless SEARCH_CACHE=true, so tests and
normal runs never read stale data by accident. Same query twice = no API call."""
import hashlib
import json
import os
import threading
import time
from pathlib import Path

CACHE_PATH = Path("data/cache/search_cache.json")
TTL_SECONDS = 24 * 3600

_lock = threading.Lock()
_stats = {"hits": 0, "misses": 0}


def enabled() -> bool:
    return os.getenv("SEARCH_CACHE", "false").lower() == "true"


def _key(kind: str, *parts) -> str:
    raw = json.dumps([kind, *[str(p).strip().lower() for p in parts]])
    return hashlib.sha1(raw.encode()).hexdigest()


def _load() -> dict:
    try:
        return json.loads(CACHE_PATH.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return {}


def get(kind: str, *parts, ttl: int = TTL_SECONDS):
    if not enabled():
        return None
    with _lock:
        entry = _load().get(_key(kind, *parts))
        if entry and time.time() - entry["t"] <= ttl:
            _stats["hits"] += 1
            return entry["v"]
        _stats["misses"] += 1
        return None


def put(kind: str, *parts, value) -> None:
    if not enabled():
        return
    with _lock:
        data = _load()
        data[_key(kind, *parts)] = {"t": time.time(), "v": value}
        CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
        tmp = CACHE_PATH.with_suffix(".tmp")
        tmp.write_text(json.dumps(data), encoding="utf-8")
        os.replace(tmp, CACHE_PATH)


def stats() -> dict:
    return dict(_stats)

Writing tools/cache.py


In [2]:
%%writefile tools/verification.py
"""Targeted corroboration (Day 25). For claims backed by fewer than the required number of
independent domains, search for the claim itself, excluding the domain(s) already cited,
and accept a new source ONLY if one of its sentences actually states the claim.
No LLM calls: this spends search requests, not Groq output tokens."""
import os
import re
from typing import Callable, Dict, List, Optional, Tuple

from tools.claim_merge import _profile
from tools.evidence import (independent_domains, independent_source_count,
                            MIN_INDEPENDENT_SOURCES)

MAX_VERIFY_CLAIMS = 6            # per run, keeps search volume inside free-tier limits
OVERLAP_WITH_NUMBERS = 0.5       # claim has figures -> the same figures must appear in the sentence
OVERLAP_NO_NUMBERS = 0.75        # no figures to anchor on -> demand much closer wording


def enabled() -> bool:
    return os.getenv("VERIFY_BY_SEARCH", "false").lower() == "true"


def _sentences(text: str) -> List[str]:
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", text or "") if s.strip()]


def supports(claim: str, text: str) -> bool:
    """True if a single sentence of `text` states the claim: same figures, same polarity,
    and enough shared content words. Sentence-level so a stray 'not' elsewhere can't flip it."""
    p = _profile(claim)
    if not p["tokens"]:
        return False
    threshold = OVERLAP_WITH_NUMBERS if p["numbers"] else OVERLAP_NO_NUMBERS
    for sentence in _sentences(text):
        s = _profile(sentence)
        if p["numbers"] and not p["numbers"] <= s["numbers"]:
            continue
        if s["negated"] != p["negated"]:
            continue
        if len(p["tokens"] & s["tokens"]) / len(p["tokens"]) >= threshold:
            return True
    return False


def verify_claim(fact: Dict, search_fn: Callable) -> Tuple[Dict, int, bool]:
    """Returns (fact, sources_added, search_failed)."""
    have = independent_domains(fact.get("sources", []))
    try:
        candidates = search_fn(fact["claim"], exclude_domains=sorted(have))
    except Exception:
        return fact, 0, True

    sources = list(fact.get("sources", []))
    added = 0
    for c in candidates or []:
        new_domain = independent_domains([c]) - have
        if new_domain and supports(fact["claim"], c.get("content", "")):
            have |= new_domain
            sources.append({"id": len(sources) + 1, "title": c.get("title", ""),
                            "url": c.get("url", ""), "via": "verification"})
            added += 1
    if not added:
        return fact, 0, False
    return {**fact, "sources": sources, "verified_by_search": added}, added, False


def verify_claims(facts: List[Dict], search_fn: Callable,
                  max_claims: int = MAX_VERIFY_CLAIMS) -> Tuple[List[Dict], Dict]:
    """Verify the most checkable under-sourced claims (those with figures first), sequentially."""
    needy = [i for i, f in enumerate(facts)
             if independent_source_count(f.get("sources", [])) < MIN_INDEPENDENT_SOURCES]
    needy.sort(key=lambda i: (not _profile(facts[i]["claim"])["numbers"], i))

    out = list(facts)
    stats = {"tried": 0, "upgraded": 0, "sources_added": 0, "failed": 0}
    for i in needy[:max_claims]:
        stats["tried"] += 1
        new_fact, added, failed = verify_claim(out[i], search_fn)
        stats["failed"] += int(failed)
        stats["sources_added"] += added
        if independent_source_count(new_fact.get("sources", [])) >= MIN_INDEPENDENT_SOURCES:
            stats["upgraded"] += 1
        out[i] = new_fact
    return out, stats

Writing tools/verification.py


In [3]:
%%writefile tools/llm_clients.py
import os
import time
import threading
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from tavily import TavilyClient
from ddgs import DDGS

from tools import cache
from tools.diversity import (select_diverse, add_new_domains, domain_count,
                             MAX_SOURCES_PER_QUERY, MIN_DIVERSE_DOMAINS)
from tools.evidence import source_domain

load_dotenv()

GROQ_KEY = os.getenv("GROQ_API_KEY")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")
DEMO_MODE = os.getenv("DEMO_MODE", "false").lower() == "true"

assert GROQ_KEY, "Missing GROQ_API_KEY"
assert GEMINI_KEY, "Missing GEMINI_API_KEY"
assert TAVILY_KEY, "Missing TAVILY_API_KEY"

reasoning_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
gemini_client = genai.Client(api_key=GEMINI_KEY)
tavily_client = TavilyClient(api_key=TAVILY_KEY)

MAX_CONCURRENT_SEARCHES = 1 if DEMO_MODE else 2
_search_semaphore = threading.Semaphore(MAX_CONCURRENT_SEARCHES)

# Day 24: fetch a wider candidate pool, but KEEP only distinct domains.
CANDIDATE_POOL = 8


def is_rate_limit_error(e: Exception) -> bool:
    msg = str(e).lower()
    return "429" in msg or "rate limit" in msg or "resource_exhausted" in msg


def call_llm(prompt: str, max_retries: int = 2) -> str:
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return reasoning_llm.invoke(prompt).content
        except Exception as e:
            last_error = e
            if is_rate_limit_error(e) and attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            raise
    raise last_error


def _mark_snippets(snippets) -> str:
    """Tags each snippet [S1], [S2]... so extraction can say which source states a claim (Day 23)."""
    return " ".join(f"[S{i}] {text[:300]}" for i, text in enumerate(snippets, start=1))


def _package(results, provider: str) -> dict:
    """results: [{"title","url","content"}] -> the dict the agents expect."""
    sources = [{"id": i, "title": r.get("title", ""), "url": r.get("url", "")}
               for i, r in enumerate(results, start=1)]
    answer = _mark_snippets([r.get("content", "") for r in results])
    return {"answer": answer, "sources": sources, "provider": provider}


def _ddg_candidates(query: str, n: int = CANDIDATE_POOL) -> list:
    raw = list(DDGS().text(query, max_results=n))
    return [{"title": r.get("title", ""), "url": r.get("href") or r.get("url", ""),
             "content": r.get("body", "")} for r in raw]


def _search_duckduckgo(query: str) -> dict:
    picked = select_diverse(_ddg_candidates(query), max_total=MAX_SOURCES_PER_QUERY)
    return _package(picked, "duckduckgo")


def _research_uncached(query: str) -> dict:
    """Gemini -> Tavily -> DuckDuckGo, concurrency-limited by a semaphore."""
    with _search_semaphore:
        try:
            resp = gemini_client.models.generate_content(
                model="gemini-3.6-flash",
                contents=query,
                config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
            )
            sources = []
            if resp.candidates[0].grounding_metadata:
                sources = [
                    {"id": i, "title": c.web.title, "url": c.web.uri}
                    for i, c in enumerate(resp.candidates[0].grounding_metadata.grounding_chunks, start=1)
                ]
            # Gemini's answer is one synthesized text, so it has no per-snippet tags.
            return {"answer": resp.text, "sources": sources, "provider": "gemini"}
        except (genai_errors.ClientError, genai_errors.ServerError) as e:
            print(f"⚠️ Gemini unavailable ({type(e).__name__}), falling back to Tavily...")

        try:
            raw = tavily_client.search(query=query, max_results=CANDIDATE_POOL)["results"]
            candidates = [{"title": r.get("title", ""), "url": r.get("url", ""),
                           "content": r.get("content", "")} for r in raw]
            picked = select_diverse(candidates, max_total=MAX_SOURCES_PER_QUERY)
            provider = "tavily"
            # Too few distinct domains? Top up from DuckDuckGo, but only with domains we don't have.
            if domain_count(picked) < MIN_DIVERSE_DOMAINS:
                try:
                    topped = add_new_domains(picked, _ddg_candidates(query), MAX_SOURCES_PER_QUERY)
                    if len(topped) > len(picked):
                        picked, provider = topped, "tavily+duckduckgo"
                except Exception as e:
                    print(f"⚠️ DuckDuckGo top-up skipped ({type(e).__name__})")
            return _package(picked, provider)
        except Exception as e:
            print(f"⚠️ Tavily unavailable ({type(e).__name__}), falling back to DuckDuckGo...")

        try:
            return _search_duckduckgo(query)
        except Exception as e:
            print(f"⚠️ DuckDuckGo also unavailable ({type(e).__name__})")
            return {"answer": "", "sources": [], "provider": "none"}


def research_with_fallback(query: str) -> dict:
    """Cached wrapper (Day 25). Caching is opt-in via SEARCH_CACHE=true; failures are never cached."""
    cached = cache.get("research", query)
    if cached is not None:
        return {**cached, "cached": True}
    result = _research_uncached(query)
    if result.get("provider") != "none" and result.get("answer"):
        cache.put("research", query, value=result)
    return result


def search_candidates(query: str, exclude_domains=(), max_total: int = MAX_SOURCES_PER_QUERY) -> list:
    """Raw, full-text search results for verifying one claim (Day 25): Tavily, then DuckDuckGo.
    Results from `exclude_domains` are dropped; one result per remaining domain is kept."""
    exclude = {d.lower() for d in exclude_domains}
    cached = cache.get("candidates", query, ",".join(sorted(exclude)))
    if cached is not None:
        return cached

    def usable(results):
        keep = [r for r in results
                if source_domain({"url": r.get("url", ""), "title": r.get("title", "")}) not in exclude]
        return select_diverse(keep, max_total=max_total) if keep else []

    found = []
    with _search_semaphore:
        try:
            kwargs = {"query": query, "max_results": CANDIDATE_POOL}
            if exclude:
                kwargs["exclude_domains"] = sorted(exclude)
            raw = tavily_client.search(**kwargs)["results"]
            found = usable([{"title": r.get("title", ""), "url": r.get("url", ""),
                             "content": r.get("content", "")} for r in raw])
        except Exception as e:
            print(f" Verification search via Tavily failed ({type(e).__name__}), trying DuckDuckGo...")
        if not found:
            try:
                found = usable(_ddg_candidates(query))
            except Exception as e:
                print(f"Verification search via DuckDuckGo failed ({type(e).__name__})")
    if found:
        cache.put("candidates", query, ",".join(sorted(exclude)), value=found)
    return found

Overwriting tools/llm_clients.py


In [4]:
p = "agents/fact_checking_agent.py"
s = open(p, encoding="utf-8").read()

s = s.replace("from tools.llm_clients import call_llm\n",
              "from tools.llm_clients import call_llm, search_candidates\n")
s = s.replace("from tools.claim_merge import merge_duplicate_claims, expand_verdicts\n",
              "from tools.claim_merge import merge_duplicate_claims, expand_verdicts\nfrom tools import verification\n")

old = s[s.index("    checked = expand_verdicts(fact_check_facts(merged))"):]
new = '''    # Day 25: for under-sourced claims, search for corroboration on OTHER domains (opt-in).
    vstats = {}
    if verification.enabled():
        merged, vstats = verification.verify_claims(merged, search_candidates)
        log_agent_event(AGENT_NAME, "verification_done", **vstats)

    checked = expand_verdicts(fact_check_facts(merged))
    return {
        "fact_check_results": checked,
        "model_usage_log": [{"agent": AGENT_NAME, "provider": "groq", "facts_checked": len(merged),
                             "facts_in": len(extracted_facts), **{f"verify_{k}": v for k, v in vstats.items()}}],
    }
'''
s = s.replace(old, new)
assert "verification.verify_claims" in s and "search_candidates" in s
open(p, "w", encoding="utf-8").write(s)
print("patched")

patched


In [5]:
p = "tools/metrics.py"
s = open(p, encoding="utf-8").read()
a = '''            "search_providers": dict(providers)}'''
b = '''            "search_providers": dict(providers),
            "claims_verified_by_search": sum(1 for f in results if f.get("verified_by_search"))}'''
c = '''    print("  search providers:             ", ev["search_providers"])'''
d = c + '''
    print("  claims given extra sources by verification search:", ev["claims_verified_by_search"])'''
assert s.count(a) == 1 and s.count(c) == 1
open(p, "w", encoding="utf-8").write(s.replace(a, b).replace(c, d))
print("patched")

patched


In [6]:
%%writefile tests/test_verification.py
from tools.verification import supports, verify_claim, verify_claims

CLAIM = "Hydrometallurgical processing achieves a 95% recovery rate for lithium."


def cand(url, text, title="t"):
    return {"title": title, "url": url, "content": text}


def test_supports_requires_same_figures():
    assert supports(CLAIM, "Hydrometallurgical processes reach a 95% recovery rate for lithium in tests.")
    assert not supports(CLAIM, "Hydrometallurgical processes reach a 70% recovery rate for lithium.")


def test_supports_is_sentence_level_for_negation():
    text = "This is not a small market. Hydrometallurgical processing achieves a 95% recovery rate for lithium."
    assert supports(CLAIM, text)           # the stray 'not' is in a different sentence
    assert not supports(CLAIM, "Hydrometallurgical processing does not achieve a 95% recovery rate for lithium.")


def test_claims_without_figures_need_close_wording():
    claim = "Redwood Materials operates the largest lithium-ion battery recycling facility outside Asia"
    assert not supports(claim, "Redwood Materials launched a second-life battery storage partnership.")
    assert supports(claim, "Redwood Materials operates the largest lithium-ion battery recycling facility outside Asia today.")


def test_verify_adds_only_new_supporting_domains():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"}]}
    calls = []

    def search(q, exclude_domains=()):
        calls.append(exclude_domains)
        return [cand("https://a.org/y", "Hydrometallurgical processing achieves a 95% recovery rate for lithium."),
                cand("https://b.com/z", "Hydrometallurgical processing achieves a 95% recovery rate for lithium."),
                cand("https://c.net/q", "Unrelated text about cathodes.")]

    new, added, failed = verify_claim(fact, search)
    assert calls == [["a.org"]] and added == 1 and not failed
    assert [s["url"] for s in new["sources"]] == ["https://a.org/x", "https://b.com/z"]
    assert new["sources"][1]["via"] == "verification" and [s["id"] for s in new["sources"]] == [1, 2]


def test_search_failure_leaves_fact_unchanged():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"}]}

    def boom(q, exclude_domains=()):
        raise RuntimeError("down")

    new, added, failed = verify_claim(fact, boom)
    assert new is fact and added == 0 and failed


def test_verify_claims_prioritises_figures_and_respects_cap():
    def mk(claim, n):
        return {"claim": claim, "sources": [{"id": 1, "title": "a", "url": f"https://s{n}.org/x"}]}
    facts = [mk("Plain statement about recycling policy and regulation across regions here", 1),
             mk("Plant capacity reaches 50000 tonnes per year in Nevada operations", 2),
             mk("Another plain statement about supply chain integration strategy overall", 3)]
    seen = []

    def search(q, exclude_domains=()):
        seen.append(q)
        return []

    _, stats = verify_claims(facts, search, max_claims=2)
    assert stats["tried"] == 2 and seen[0].startswith("Plant capacity")


def test_already_corroborated_claims_not_searched():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"},
                                         {"id": 2, "title": "b", "url": "https://b.com/x"}]}
    _, stats = verify_claims([fact], lambda q, exclude_domains=(): 1 / 0)
    assert stats["tried"] == 0


def test_upgraded_counted_when_claim_reaches_two_domains():
    fact = {"claim": CLAIM, "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"}]}
    search = lambda q, exclude_domains=(): [cand("https://b.com/z", CLAIM)]
    out, stats = verify_claims([fact], search)
    assert stats == {"tried": 1, "upgraded": 1, "sources_added": 1, "failed": 0}
    %%writefile tests/test_cache.py
import time
from tools import cache


def test_roundtrip_and_ttl(tmp_path, monkeypatch):
    monkeypatch.setattr(cache, "CACHE_PATH", tmp_path / "c.json")
    monkeypatch.setenv("SEARCH_CACHE", "true")
    assert cache.get("search", "Query A") is None
    cache.put("search", "Query A", value={"answer": "x"})
    assert cache.get("search", "  query a ") == {"answer": "x"}      # key is normalised
    assert cache.get("search", "query a", ttl=-1) is None            # expired


def test_disabled_by_default(tmp_path, monkeypatch):
    monkeypatch.setattr(cache, "CACHE_PATH", tmp_path / "c.json")
    monkeypatch.delenv("SEARCH_CACHE", raising=False)
    cache.put("search", "q", value=1)
    assert cache.get("search", "q") is None and not (tmp_path / "c.json").exists()


def test_corrupt_file_is_ignored(tmp_path, monkeypatch):
    p = tmp_path / "c.json"
    p.write_text("{not json")
    monkeypatch.setattr(cache, "CACHE_PATH", p)
    monkeypatch.setenv("SEARCH_CACHE", "true")
    assert cache.get("search", "q") is None
    cache.put("search", "q", value=2)
    assert cache.get("search", "q") == 2

Writing tests/test_verification.py


In [7]:
%%writefile tests/test_fact_check_verification.py
import json
import agents.fact_checking_agent as fc

CLAIM = "Hydrometallurgical processing achieves a 95% recovery rate for lithium."


def _facts():
    return [{"claim": CLAIM, "sub_question": "Q1",
             "sources": [{"id": 1, "title": "a", "url": "https://a.org/x"}]}]


def test_verification_upgrades_claim_to_confirmed(monkeypatch):
    monkeypatch.setenv("VERIFY_BY_SEARCH", "true")
    monkeypatch.setattr(fc, "search_candidates", lambda q, exclude_domains=(): [
        {"title": "b", "url": "https://b.com/z", "content": CLAIM}])
    monkeypatch.setattr(fc, "call_llm", lambda p: json.dumps(
        [{"index": 0, "status": "CONFIRMED", "reasoning": "two domains"}]))
    out = fc.fact_checking_node({"extracted_facts": _facts()})
    r = out["fact_check_results"][0]
    assert r["status"] == "CONFIRMED" and r["independent_sources"] == 2 and r["verified_by_search"] == 1
    assert out["model_usage_log"][0]["verify_upgraded"] == 1


def test_verification_off_by_default_never_searches(monkeypatch):
    monkeypatch.delenv("VERIFY_BY_SEARCH", raising=False)
    monkeypatch.setattr(fc, "search_candidates", lambda *a, **k: 1 / 0)
    monkeypatch.setattr(fc, "call_llm", lambda p: json.dumps(
        [{"index": 0, "status": "UNVERIFIED", "reasoning": "one source"}]))
    r = fc.fact_checking_node({"extracted_facts": _facts()})["fact_check_results"][0]
    assert r["status"] == "UNVERIFIED"

Writing tests/test_fact_check_verification.py


In [1]:
!python -m pytest tests/ -q
with open(".gitignore", "a") as f:
    f.write("\ndata/cache/\n")

........................................................................ [ 71%]
.............................                                            [100%]
101 passed in 8.87s


In [2]:
from pathlib import Path

p = Path("tests/test_verification.py")
raw = p.read_text(encoding="utf-8-sig")

files, current, buf = {}, "tests/test_verification.py", []
for line in raw.splitlines(keepends=True):
    stripped = line.strip().lstrip("\ufeff")
    if stripped.startswith("%%writefile"):
        files[current] = "".join(buf)
        current = stripped.split(None, 1)[1].strip().replace("\\", "/")
        buf = []
    else:
        buf.append(line)
files[current] = "".join(buf)

for name, body in files.items():
    compile(body, name, "exec")          # stops here if any file is still invalid
for name, body in files.items():
    Path(name).write_text(body, encoding="utf-8")
    print("wrote", name)

wrote tests/test_verification.py
wrote tests/test_cache.py


In [2]:
import os
os.environ["VERIFY_BY_SEARCH"] = "true"
os.environ["SEARCH_CACHE"] = "true"

from graph.build_graph import build_graph
from tools.metrics import RunMetrics, save_run, print_run_report
from tools import cache

app = build_graph()
metrics = RunMetrics()
query = "Compare battery recycling approaches of Tesla, Redwood Materials and CATL"
initial_state = {
    "query": query, "plan": [], "next_agent": "", "iteration_count": 0,
    "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
    "model_usage_log": [],
}
result = app.invoke(initial_state, config={"callbacks": [metrics]})
print_run_report(metrics, result)
print("cache:", cache.stats())
print("saved:", save_run(metrics, result, label="day25"))

2026-11-03 15:44:34,201 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=4
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


⚠️ Gemini unavailable (ClientError), falling back to Tavily...
⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-11-03 15:44:38,381 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0


⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-11-03 15:44:41,002 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0


⚠️ Gemini unavailable (ClientError), falling back to Tavily...


2026-11-03 15:44:41,695 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 15:44:43,427 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 15:44:43,432 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-11-03 15:44:44,814 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the primary technical processes and recovery rates for lithium, cobalt, and nickel in the recycling facilities operated by Tesla, Redwood Materials, and CATL? facts_found=13
2026-11-03 15:44:45,957 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=How do the business models and supply chain integration strategies of Tesla, Redwood Materials, and CATL differ in sourcing end-of-life batteries for recycling? facts_found=12
2026-11-03 15:45:24,349 | INFO | research_system | [data_extra

Total: 227s | output tokens: 5057 (reasoning: 0) | 429s: 1
Peak output tokens in any 60s window: 1802 (Groq limit: 1000)

node                   calls    out  reason max/call  429  node_s
supervisor                 1    127       0      127    0     0.8
data_extraction            5   1926       0      604    1    69.6
fact_checking              3   1359       0      510    0    82.6
summarization              4    700       0      195    0    48.1
report_generator           1    945       0      945    0    16.5
web_search_single          0      0       0        0    0    27.6

Evidence
  claims extracted: 44 -> after merge/expand: 39 (3 rows from merged groups)
  cited sources per claim:       {1: 32, 2: 3, 3: 2, 5: 2}
  independent domains per claim: {1: 32, 3: 2, 2: 3, 5: 2}
  status:                        {'UNVERIFIED': 32, 'CONFIRMED': 7}
  search providers:              {'tavily': 4}
  claims given extra sources by verification search: 5
cache: {'hits': 0, 'misses': 10}
saved: d

In [3]:
import os
os.environ["VERIFY_BY_SEARCH"] = "true"
os.environ["SEARCH_CACHE"] = "true"

from graph.build_graph import build_graph
from tools.metrics import RunMetrics, save_run, print_run_report
from tools import cache

app = build_graph()
metrics = RunMetrics()
query = "Compare battery recycling approaches of Tesla, Redwood Materials and CATL"
initial_state = {
    "query": query, "plan": [], "next_agent": "", "iteration_count": 0,
    "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
    "model_usage_log": [],
}
result = app.invoke(initial_state, config={"callbacks": [metrics]})
print_run_report(metrics, result)
print("cache:", cache.stats())
print("saved:", save_run(metrics, result, label="day25"))

2026-11-03 16:03:27,855 | INFO | research_system | [supervisor_agent] plan_generated sub_questions=4
2026-11-03 16:03:27,876 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 16:03:27,879 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 16:03:27,884 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 16:03:27,888 | INFO | research_system | [web_search_agent] search_done status=ok provider=tavily sources=4 retries=0
2026-11-03 16:03:27,893 | INFO | research_system | [supervisor_agent] routing decision=data_extraction iteration=2
2026-11-03 16:03:29,433 | INFO | research_system | [data_extraction_agent] extraction_done sub_question=What are the primary technical processes and recovery rates for lithium, cobalt, and nickel in the recycling facilities operated by Tesla, Redwood Materials, and CATL? 

 Verification search via Tavily failed (ConnectionError), trying DuckDuckGo...


2026-11-03 16:04:37,012 | INFO | research_system | [fact_checking_agent] verification_done tried=6 upgraded=4 sources_added=13 failed=0
2026-11-03 16:04:42,273 | INFO | research_system | [fact_checking_agent] batch_done downgraded=0 UNVERIFIED=10 CONFIRMED=5
2026-11-03 16:05:03,986 | INFO | research_system | [fact_checking_agent] batch_done downgraded=0 CONFIRMED=1 UNVERIFIED=14
2026-11-03 16:05:41,370 | INFO | research_system | [fact_checking_agent] batch_done downgraded=0 UNVERIFIED=1
2026-11-03 16:05:41,377 | INFO | research_system | [supervisor_agent] routing decision=summarization iteration=4
2026-11-03 16:05:45,186 | INFO | research_system | [summarization_agent] summary_done sub_question=What are the primary technical processes and recovery rates for lithium, cobalt, and nickel in the recycling facilities operated by Tesla, Redwood Materials, and CATL? status=ok CONFIRMED=1 DISPUTED=0 UNVERIFIED=4
2026-11-03 16:05:52,941 | INFO | research_system | [summarization_agent] summary_d

Total: 175s | output tokens: 4676 (reasoning: 0) | 429s: 0
Peak output tokens in any 60s window: 2077 (Groq limit: 1000)

node                   calls    out  reason max/call  429  node_s
supervisor                 1    127       0      127    0     0.6
data_extraction            4   1950       0      615    0    58.2
fact_checking              3   1162       0      616    0    75.2
summarization              4    583       0      175    0    32.9
report_generator           1    854       0      854    0     8.1
web_search_single          0      0       0        0    0     0.1

Evidence
  claims extracted: 36 -> after merge/expand: 31 (3 rows from merged groups)
  cited sources per claim:       {1: 25, 2: 2, 3: 1, 4: 1, 5: 2}
  independent domains per claim: {1: 25, 3: 1, 2: 2, 5: 2, 4: 1}
  status:                        {'UNVERIFIED': 25, 'CONFIRMED': 6}
  search providers:              {'tavily': 4}
  claims given extra sources by verification search: 4
cache: {'hits': 6, 'misses': 

In [ ]:
os.environ["SEARCH_CACHE"] = "true"

from graph.build_graph import build_graph
from tools.metrics import RunMetrics, save_run, print_run_report
from tools import cache

app = build_graph()
metrics = RunMetrics()
query = "Compare battery recycling approaches of Tesla, Redwood Materials and CATL"
initial_state = {
    "query": query, "plan": [], "next_agent": "", "iteration_count": 0,
    "needs_more_research": False, "search_results": [], "extracted_facts": [],
    "fact_check_results": [], "summaries": [], "errors": [], "final_report": None,
    "model_usage_log": [],
}
result = app.invoke(initial_state, config={"callbacks": [metrics]})
print_run_report(metrics, result)
print("cache:", cache.stats())
print("saved:", save_run(metrics, result, label="day25"))